In [1]:

import sys
import subprocess
import json
import sqlite3
import shutil
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "pandas", "numpy", "reportlab", "pyarrow"
    ])
    import pandas as pd
    import numpy as np

    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT DISCOVERY
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
PREPARED_ROOT = PROJECT_ROOT / "data" / "prepared"
REPORTS_DIR = PROJECT_ROOT / "reports"
VALIDATION_DIR = REPORTS_DIR / "validation"
LOGS_DIR = PROJECT_ROOT / "logs"
SRC_DIR = PROJECT_ROOT / "src"

WAREHOUSE_DIR = PROJECT_ROOT / "warehouse"
WAREHOUSE_DIR.mkdir(parents=True, exist_ok=True)

TRANSFORMED_DIR = PROJECT_ROOT / "data" / "transformed"
TRANSFORMED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE_NAME = "06 Feature Engineering and Transformation- DM4ML-Group51.pdf"
OUTPUT_PATH = PROJECT_ROOT / OUTPUT_FILE_NAME

SQLITE_DB_PATH = WAREHOUSE_DIR / "recommendation_features.db"
SCHEMA_SQL_PATH = WAREHOUSE_DIR / "recommendation_features_schema.sql"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_ROOT: {RAW_ROOT}")
print(f"BRONZE_ROOT: {BRONZE_ROOT}")
print(f"WAREHOUSE DB: {SQLITE_DB_PATH}")
print(f"SCHEMA SQL: {SCHEMA_SQL_PATH}")
print(f"OUTPUT PDF: {OUTPUT_PATH}")

# ============================================================
# 2) HELPERS
# ============================================================
def safe_str(x):
    try:
        return str(x)
    except Exception:
        return ""

def rel_path(path):
    try:
        return safe_str(path.relative_to(PROJECT_ROOT))
    except Exception:
        return safe_str(path)

def collect_files(base_dir, patterns):
    results = []
    if not base_dir.exists():
        return results
    for pattern in patterns:
        results.extend(base_dir.rglob(pattern))
    return sorted(set(p for p in results if p.is_file()))

def latest_file(base_dir, patterns):
    files = collect_files(base_dir, patterns)
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

def file_info(path):
    if not path or not path.exists():
        return None
    st = path.stat()
    return {
        "name": path.name,
        "relative_path": rel_path(path),
        "size_kb": round(st.st_size / 1024, 2),
        "modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "suffix": path.suffix.lower(),
    }

def read_text_preview(path, max_lines=60, max_chars=8000):
    if not path or not path.exists():
        return "File not found."
    try:
        text = path.read_text(encoding="utf-8", errors="ignore")
        lines = text.splitlines()[:max_lines]
        return "\n".join(lines)[:max_chars]
    except Exception as e:
        return f"Could not preview file: {e}"

def read_notebook_preview(path, max_code_cells=4, max_chars=7000):
    if not path or not path.exists():
        return "Notebook not found."
    try:
        with open(path, "r", encoding="utf-8") as f:
            nb = json.load(f)
        blocks = []
        code_idx = 0
        for cell in nb.get("cells", []):
            if cell.get("cell_type") != "code":
                continue
            code_idx += 1
            src = cell.get("source", [])
            src = "".join(src) if isinstance(src, list) else str(src)
            src = src.strip()
            if src:
                blocks.append(f"# Code cell {code_idx}\n{src}")
            if len("\n\n".join(blocks)) >= max_chars or code_idx >= max_code_cells:
                break
        out = "\n\n".join(blocks).strip()
        return out[:max_chars] if out else "No code cells found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def read_code_preview(path, max_lines=120, max_chars=7000):
    if not path or not path.exists():
        return "File not found."
    if path.suffix.lower() == ".ipynb":
        return read_notebook_preview(path, max_code_cells=4, max_chars=max_chars)
    return read_text_preview(path, max_lines=max_lines, max_chars=max_chars)

def build_tree_text(base_path, max_depth=5, max_items=250):
    if not base_path.exists():
        return f"{base_path.name}/ (not found)"
    lines = [f"{base_path.name}/"]
    count = 0

    def walk(path, prefix="", depth=0):
        nonlocal count
        if depth >= max_depth or count >= max_items:
            return
        items = sorted(path.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
        for idx, item in enumerate(items):
            if count >= max_items:
                break
            connector = "└── " if idx == len(items) - 1 else "├── "
            lines.append(prefix + connector + item.name + ("/" if item.is_dir() else ""))
            count += 1
            if item.is_dir():
                extension = "    " if idx == len(items) - 1 else "│   "
                walk(item, prefix + extension, depth + 1)

    walk(base_path)
    if count >= max_items:
        lines.append("... output truncated ...")
    return "\n".join(lines)

def wrap_block_text(text, width=95):
    wrapped = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped.append("")
            continue
        parts = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped.extend(parts if parts else [""])
    return "\n".join(wrapped)

def make_hashable_value(v):
    if isinstance(v, np.ndarray):
        return tuple(make_hashable_value(x) for x in v.tolist())
    if isinstance(v, list):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, tuple):
        return tuple(make_hashable_value(x) for x in v)
    if isinstance(v, set):
        return tuple(sorted(make_hashable_value(x) for x in v))
    if isinstance(v, dict):
        return json.dumps(v, sort_keys=True, ensure_ascii=False, default=str)
    try:
        hash(v)
        return v
    except TypeError:
        return str(v)

def make_display_value(v):
    if isinstance(v, np.ndarray):
        return json.dumps(v.tolist(), ensure_ascii=False)
    if isinstance(v, (list, tuple, set)):
        try:
            return json.dumps(list(v), ensure_ascii=False)
        except Exception:
            return str(v)
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True, default=str)
        except Exception:
            return str(v)
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return safe_str(v)

def wrap_path_for_pdf(value, max_chunk=32):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=36):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, header_bg="#D9EAD3", path_cols=None, file_cols=None):
    path_cols = path_cols or []
    file_cols = file_cols or []
    converted = []

    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            style = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), style))
            else:
                if c in path_cols or c in file_cols:
                    row_cells.append(to_para(cell, style, kind="path"))
                else:
                    row_cells.append(to_para(cell, style, kind="general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor(header_bg)),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=20, col_widths=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        preview = df.head(max_rows).copy()
        for col in preview.columns:
            preview[col] = preview[col].map(make_display_value)
        data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths)

def safe_duplicate_count(df):
    if df is None or df.empty:
        return 0
    tmp = df.copy()
    for col in tmp.columns:
        tmp[col] = tmp[col].map(make_hashable_value)
    return int(tmp.duplicated().sum())

def load_table_file(path, nrows=None):
    if not path or not path.exists():
        return None
    try:
        suffix = path.suffix.lower()
        if suffix == ".csv":
            return pd.read_csv(path, nrows=nrows)
        if suffix == ".parquet":
            df = pd.read_parquet(path)
            return df.head(nrows) if nrows else df
        if suffix == ".json":
            with open(path, "r", encoding="utf-8") as f:
                payload = json.load(f)
            if isinstance(payload, list):
                return pd.DataFrame(payload).head(nrows) if nrows else pd.DataFrame(payload)
            if isinstance(payload, dict):
                for v in payload.values():
                    if isinstance(v, list) and v and isinstance(v[0], dict):
                        df = pd.DataFrame(v)
                        return df.head(nrows) if nrows else df
                return pd.DataFrame([payload]).head(nrows) if nrows else pd.DataFrame([payload])
    except Exception as e:
        print(f"Could not load {path}: {e}")
        return None
    return None

# ============================================================
# 3) DISCOVERY HELPERS
# ============================================================
def discover_events_file():
    patterns = [
        "**/events.csv",
        "**/*events*.csv",
        "**/*interaction*.csv",
        "**/*clickstream*.csv",
        "**/*transactions*.csv",
    ]
    matches = []
    for pattern in patterns:
        if RAW_ROOT.exists():
            matches.extend(RAW_ROOT.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_products_file():
    patterns = [
        "**/products.parquet",
        "**/products.csv",
        "**/products_raw.json",
        "**/*product*.parquet",
        "**/*product*.csv",
        "**/*product*.json",
    ]
    matches = []
    for base in [BRONZE_ROOT, RAW_ROOT]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def discover_prepared_interactions_file():
    patterns = [
        "**/prepared_interactions*.parquet",
        "**/prepared_interactions*.csv",
        "**/*prepared*interaction*.parquet",
        "**/*prepared*interaction*.csv",
    ]
    matches = []
    for base in [PREPARED_ROOT, PROJECT_ROOT / "data" / "processed", PROJECT_ROOT / "data" / "silver"]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))
    return max(matches, key=lambda p: p.stat().st_mtime) if matches else None

def find_feature_assets():
    patterns = [
        "*feature*.py", "*feature*.ipynb", "*feature*.sql",
        "*transform*.py", "*transform*.ipynb", "*transform*.sql",
        "*warehouse*.py", "*warehouse*.sql",
        "*schema*.sql", "*schema*.py",
    ]
    matches = []
    for base in [PROJECT_ROOT, SRC_DIR]:
        if base.exists():
            for pattern in patterns:
                matches.extend(base.rglob(pattern))

    cleaned = []
    for p in sorted(set(matches)):
        p_str = safe_str(p).lower()
        if ".ipynb_checkpoints" in p_str:
            continue
        if "/venv/" in p_str or "\\venv\\" in p_str or "/.venv/" in p_str or "\\.venv\\" in p_str:
            continue
        if "/site-packages/" in p_str or "\\site-packages\\" in p_str:
            continue
        cleaned.append(p)
    return cleaned

# ============================================================
# 4) FEATURE PREPARATION
# ============================================================
def standardize_events(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()

    rename_map = {}
    lower_map = {c.lower(): c for c in df.columns}

    if "visitorid" in lower_map:
        rename_map[lower_map["visitorid"]] = "user_id"
    elif "userid" in lower_map:
        rename_map[lower_map["userid"]] = "user_id"

    if "itemid" in lower_map:
        rename_map[lower_map["itemid"]] = "item_id"
    elif "productid" in lower_map:
        rename_map[lower_map["productid"]] = "item_id"

    if "timestamp" in lower_map:
        rename_map[lower_map["timestamp"]] = "event_ts"
    elif "event_ts" in lower_map:
        rename_map[lower_map["event_ts"]] = "event_ts"

    if "event" in lower_map:
        rename_map[lower_map["event"]] = "event_type"
    elif "eventtype" in lower_map:
        rename_map[lower_map["eventtype"]] = "event_type"

    if "rating" in lower_map:
        rename_map[lower_map["rating"]] = "rating"
    if "score" in lower_map and "rating" not in rename_map.values():
        rename_map[lower_map["score"]] = "rating"

    df = df.rename(columns=rename_map)
    notes.append("Standardized common interaction columns where present.")

    needed = [c for c in ["user_id", "item_id"] if c in df.columns]
    if needed:
        before = len(df)
        df = df.dropna(subset=needed)
        notes.append(f"Dropped rows missing required interaction keys: {before - len(df)} removed.")

    if "event_type" in df.columns:
        df["event_type"] = df["event_type"].astype(str).str.strip().str.lower()
    else:
        df["event_type"] = "interaction"

    if "event_ts" in df.columns:
        ts_num = pd.to_numeric(df["event_ts"], errors="coerce")
        if ts_num.notna().sum() > 0:
            median_val = ts_num.dropna().median()
            if median_val > 1e12:
                df["event_ts"] = pd.to_datetime(ts_num, unit="ms", errors="coerce")
            elif median_val > 1e9:
                df["event_ts"] = pd.to_datetime(ts_num, unit="s", errors="coerce")
            else:
                df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
        else:
            df["event_ts"] = pd.to_datetime(df["event_ts"], errors="coerce")
    else:
        df["event_ts"] = pd.NaT

    if "rating" in df.columns:
        df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

    event_weight_map = {
        "view": 1.0,
        "click": 1.0,
        "interaction": 1.0,
        "addtocart": 3.0,
        "cart": 3.0,
        "purchase": 5.0,
        "transaction": 5.0,
    }
    df["interaction_weight"] = df["event_type"].map(event_weight_map).fillna(1.0)

    dedupe_cols = [c for c in ["user_id", "item_id", "event_type", "event_ts"] if c in df.columns]
    if dedupe_cols:
        before = len(df)
        df = df.drop_duplicates(subset=dedupe_cols)
        notes.append(f"Removed duplicate interactions using {', '.join(dedupe_cols)}: {before - len(df)} removed.")

    return df, notes

def standardize_products(df):
    if df is None or df.empty:
        return None, []

    notes = []
    df = df.copy()

    for col in df.columns:
        df[col] = df[col].map(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

    lower_map = {c.lower(): c for c in df.columns}
    rename_map = {}

    if "id" in lower_map and "product_id" not in df.columns:
        rename_map[lower_map["id"]] = "product_id"
    if "title" in lower_map:
        rename_map[lower_map["title"]] = "title"
    if "category" in lower_map:
        rename_map[lower_map["category"]] = "category"
    if "price" in lower_map:
        rename_map[lower_map["price"]] = "price"

    df = df.rename(columns=rename_map)
    notes.append("Standardized common product columns where present.")

    if "category" in df.columns:
        df["category"] = df["category"].astype("string").fillna("unknown").str.strip().str.lower()

    if "price" in df.columns:
        df["price"] = pd.to_numeric(df["price"], errors="coerce")
        median_price = df["price"].median()
        if pd.notna(median_price):
            df["price"] = df["price"].fillna(median_price)
        pmin, pmax = df["price"].min(), df["price"].max()
        if pd.notna(pmin) and pd.notna(pmax) and pmax != pmin:
            df["price_norm"] = (df["price"] - pmin) / (pmax - pmin)
        else:
            df["price_norm"] = 0.0
        notes.append("Prepared product price fields for downstream feature joins.")

    if "product_id" in df.columns:
        before = len(df)
        product_key = df["product_id"].map(make_hashable_value)
        df = df.loc[~product_key.duplicated()].copy()
        notes.append(f"Removed duplicate product_id rows: {before - len(df)} removed.")

    return df, notes

def build_feature_tables(events_df, products_df):
    outputs = {
        "user_features": pd.DataFrame(),
        "item_features": pd.DataFrame(),
        "user_item_features": pd.DataFrame(),
        "item_cooccurrence": pd.DataFrame(),
        "feature_logic": [],
    }

    if events_df is None or events_df.empty:
        outputs["feature_logic"].append("No interaction dataset was available, so feature tables could not be created.")
        return outputs

    if not {"user_id", "item_id"}.issubset(set(events_df.columns)):
        outputs["feature_logic"].append("Interaction data did not contain user_id and item_id, so recommendation features could not be created.")
        return outputs

    work = events_df.copy()

    # user activity frequency
    user_features = (
        work.groupby("user_id", dropna=False)
        .agg(
            activity_frequency=("item_id", "count"),
            unique_items_interacted=("item_id", pd.Series.nunique),
            avg_interaction_weight=("interaction_weight", "mean"),
        )
        .reset_index()
    )

    if "rating" in work.columns and work["rating"].notna().sum() > 0:
        user_rating = work.groupby("user_id", dropna=False)["rating"].mean().reset_index(name="avg_rating_per_user")
        item_rating = work.groupby("item_id", dropna=False)["rating"].mean().reset_index(name="avg_rating_per_item")
        rating_logic = "Average rating features were computed from the rating column."
    else:
        user_rating = work.groupby("user_id", dropna=False)["interaction_weight"].mean().reset_index(name="avg_rating_per_user")
        item_rating = work.groupby("item_id", dropna=False)["interaction_weight"].mean().reset_index(name="avg_rating_per_item")
        rating_logic = "No explicit rating column was available, so weighted interaction score was used as a proxy average rating."

    user_features = user_features.merge(user_rating, on="user_id", how="left")
    user_features["activity_frequency"] = user_features["activity_frequency"].astype(int)
    user_features["unique_items_interacted"] = user_features["unique_items_interacted"].astype(int)

    # item features
    item_features = (
        work.groupby("item_id", dropna=False)
        .agg(
            item_interaction_count=("user_id", "count"),
            unique_users=("user_id", pd.Series.nunique),
            avg_interaction_weight=("interaction_weight", "mean"),
        )
        .reset_index()
        .merge(item_rating, on="item_id", how="left")
    )

    if products_df is not None and not products_df.empty:
        join_df = products_df.copy()
        if "product_id" in join_df.columns:
            join_df["product_id"] = join_df["product_id"].astype(str)
            item_features["item_id"] = item_features["item_id"].astype(str)
            join_cols = ["product_id"]
            for c in ["title", "category", "price", "price_norm"]:
                if c in join_df.columns:
                    join_cols.append(c)
            item_features = item_features.merge(
                join_df[join_cols],
                left_on="item_id",
                right_on="product_id",
                how="left"
            )
            if "product_id" in item_features.columns:
                item_features = item_features.drop(columns=["product_id"])

    # user-item features
    user_item_features = (
        work.groupby(["user_id", "item_id"], dropna=False)
        .agg(
            interaction_count=("event_type", "count"),
            total_interaction_weight=("interaction_weight", "sum"),
            last_event_ts=("event_ts", "max"),
        )
        .reset_index()
    )

    event_counts = pd.crosstab(
        [work["user_id"], work["item_id"]],
        work["event_type"]
    ).reset_index()

    user_item_features = user_item_features.merge(
        event_counts,
        on=["user_id", "item_id"],
        how="left"
    )

    # co-occurrence feature
    pairs = work[["user_id", "item_id"]].drop_duplicates()
    pair_rows = []

    grouped = pairs.groupby("user_id")["item_id"].apply(list)
    for _, item_list in grouped.items():
        item_list = [str(x) for x in item_list if pd.notna(x)]
        item_list = sorted(set(item_list))
        for i in range(len(item_list)):
            for j in range(i + 1, len(item_list)):
                pair_rows.append((item_list[i], item_list[j], 1))

    if pair_rows:
        item_cooccurrence = pd.DataFrame(pair_rows, columns=["item_id_a", "item_id_b", "cooccurrence_count"])
        item_cooccurrence = (
            item_cooccurrence.groupby(["item_id_a", "item_id_b"], as_index=False)["cooccurrence_count"]
            .sum()
            .sort_values(["cooccurrence_count", "item_id_a", "item_id_b"], ascending=[False, True, True])
        )
    else:
        item_cooccurrence = pd.DataFrame(columns=["item_id_a", "item_id_b", "cooccurrence_count"])

    outputs["user_features"] = user_features
    outputs["item_features"] = item_features
    outputs["user_item_features"] = user_item_features
    outputs["item_cooccurrence"] = item_cooccurrence

    outputs["feature_logic"] = [
        "User activity frequency = count of interactions per user.",
        rating_logic,
        "Item popularity = count of interactions and distinct users per item.",
        "User-item features = interaction count, total interaction weight, latest event timestamp, and event-type counts.",
        "Co-occurrence features = pair counts for items appearing under the same user history.",
        "Transformed tables are persisted to a structured SQLite warehouse for reporting and downstream use.",
    ]
    return outputs

def save_feature_tables_to_sqlite(feature_tables, db_path):
    db_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(db_path)

    feature_tables["user_features"].to_sql("user_features", conn, if_exists="replace", index=False)
    feature_tables["item_features"].to_sql("item_features", conn, if_exists="replace", index=False)
    feature_tables["user_item_features"].to_sql("user_item_features", conn, if_exists="replace", index=False)
    feature_tables["item_cooccurrence"].to_sql("item_cooccurrence", conn, if_exists="replace", index=False)

    conn.execute("CREATE INDEX IF NOT EXISTS idx_user_features_user_id ON user_features(user_id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_item_features_item_id ON item_features(item_id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_user_item_user_item ON user_item_features(user_id, item_id)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_item_cooccurrence_pair ON item_cooccurrence(item_id_a, item_id_b)")
    conn.commit()
    conn.close()

def export_sqlite_schema(db_path, schema_path):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
        SELECT sql
        FROM sqlite_master
        WHERE type IN ('table', 'index')
          AND name NOT LIKE 'sqlite_%'
        ORDER BY type, name
    """)
    statements = [row[0] for row in cur.fetchall() if row[0]]
    conn.close()

    schema_text = ";\n\n".join(statements).strip()
    if schema_text:
        schema_text = schema_text + ";"
    else:
        schema_text = "-- No schema objects found."
    schema_path.write_text(schema_text, encoding="utf-8")
    return schema_text

def save_feature_csvs(feature_tables):
    outputs = []
    mapping = {
        "user_features": TRANSFORMED_DIR / "user_features.csv",
        "item_features": TRANSFORMED_DIR / "item_features.csv",
        "user_item_features": TRANSFORMED_DIR / "user_item_features.csv",
        "item_cooccurrence": TRANSFORMED_DIR / "item_cooccurrence.csv",
    }
    for key, out_path in mapping.items():
        df = feature_tables.get(key)
        if df is not None and not df.empty:
            df.to_csv(out_path, index=False)
            outputs.append(out_path)
    return outputs

# ============================================================
# 5) STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.4,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.1,
    leading=9.3,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.5,
    alignment=TA_LEFT,
)

# ============================================================
# 6) DISCOVER INPUTS / LOGS / ASSETS
# ============================================================
events_file = discover_events_file()
products_file = discover_products_file()
prepared_interactions_file = discover_prepared_interactions_file()

feature_assets = find_feature_assets()
latest_validation_txt = latest_file(PROJECT_ROOT, ["**/run_validation.txt"])
latest_validation_json = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.json"])
latest_validation_pdf = latest_file(VALIDATION_DIR, ["**/data_quality_report_*.pdf"])
latest_validation_log = latest_file(LOGS_DIR, ["**/validation_log_*.jsonl"])

events_df = load_table_file(prepared_interactions_file) if prepared_interactions_file else load_table_file(events_file)
products_df = load_table_file(products_file)

events_df, event_notes = standardize_events(events_df)
products_df, product_notes = standardize_products(products_df)

feature_tables = build_feature_tables(events_df, products_df)
save_feature_tables_to_sqlite(feature_tables, SQLITE_DB_PATH)
transformed_csv_paths = save_feature_csvs(feature_tables)
schema_sql_text = export_sqlite_schema(SQLITE_DB_PATH, SCHEMA_SQL_PATH)

schema_preview = read_text_preview(SCHEMA_SQL_PATH, max_lines=200, max_chars=12000)
validation_preview = read_text_preview(latest_validation_txt, max_lines=70, max_chars=7000)
validation_json_preview = read_text_preview(latest_validation_json, max_lines=60, max_chars=7000) if latest_validation_json else "No validation JSON report found."
validation_log_preview = read_text_preview(latest_validation_log, max_lines=60, max_chars=7000) if latest_validation_log else "No validation event log found."

asset_previews = []
for p in feature_assets[:5]:
    asset_previews.append({
        "name": p.name,
        "relative_path": rel_path(p),
        "preview": read_code_preview(p, max_lines=120, max_chars=7000),
    })

warehouse_tree = build_tree_text(WAREHOUSE_DIR, max_depth=4, max_items=120)
transformed_tree = build_tree_text(TRANSFORMED_DIR, max_depth=4, max_items=120)

# ============================================================
# 7) SUMMARY TABLES
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [events_file, prepared_interactions_file, products_file, latest_validation_txt, latest_validation_json, latest_validation_pdf, latest_validation_log, SQLITE_DB_PATH, SCHEMA_SQL_PATH]:
    if p and Path(p).exists():
        info = file_info(Path(p))
        artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(artifact_rows) == 1:
    artifact_rows.append(["No artifacts found", "-", "-", "-"])

asset_rows = [["Transformation Asset", "Relative Path", "Last Modified", "Size"]]
for p in feature_assets[:20]:
    info = file_info(p)
    asset_rows.append([
        info["name"],
        info["relative_path"],
        info["modified"],
        f"{info['size_kb']} KB",
    ])
if len(asset_rows) == 1:
    asset_rows.append(["No transformation asset found", "-", "-", "-"])

transformed_rows = [["Transformed Output", "Relative Path", "Type", "Last Modified", "Size"]]
for p in transformed_csv_paths + [SQLITE_DB_PATH, SCHEMA_SQL_PATH]:
    if p and Path(p).exists():
        info = file_info(Path(p))
        transformed_rows.append([
            info["name"],
            info["relative_path"],
            info["suffix"] or "N/A",
            info["modified"],
            f"{info['size_kb']} KB",
        ])
if len(transformed_rows) == 1:
    transformed_rows.append(["No transformed outputs found", "-", "-", "-", "-"])

feature_summary_df = pd.DataFrame([
    {
        "table_name": "user_features",
        "rows": 0 if feature_tables["user_features"].empty else len(feature_tables["user_features"]),
        "columns": 0 if feature_tables["user_features"].empty else len(feature_tables["user_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["user_features"]),
    },
    {
        "table_name": "item_features",
        "rows": 0 if feature_tables["item_features"].empty else len(feature_tables["item_features"]),
        "columns": 0 if feature_tables["item_features"].empty else len(feature_tables["item_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["item_features"]),
    },
    {
        "table_name": "user_item_features",
        "rows": 0 if feature_tables["user_item_features"].empty else len(feature_tables["user_item_features"]),
        "columns": 0 if feature_tables["user_item_features"].empty else len(feature_tables["user_item_features"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["user_item_features"]),
    },
    {
        "table_name": "item_cooccurrence",
        "rows": 0 if feature_tables["item_cooccurrence"].empty else len(feature_tables["item_cooccurrence"]),
        "columns": 0 if feature_tables["item_cooccurrence"].empty else len(feature_tables["item_cooccurrence"].columns),
        "duplicate_rows": safe_duplicate_count(feature_tables["item_cooccurrence"]),
    },
])

team_table = make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch])

artifact_table = make_wrapped_table(
    artifact_rows,
    col_widths=[1.55 * inch, 3.35 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

asset_table = make_wrapped_table(
    asset_rows,
    col_widths=[1.65 * inch, 3.25 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

transformed_table = make_wrapped_table(
    transformed_rows,
    col_widths=[1.55 * inch, 2.95 * inch, 0.65 * inch, 1.05 * inch, 0.60 * inch],
    path_cols=[1],
    file_cols=[0]
)

feature_summary_table = df_to_wrapped_table(feature_summary_df, max_rows=10)
user_features_preview_table = df_to_wrapped_table(feature_tables["user_features"], max_rows=15)
item_features_preview_table = df_to_wrapped_table(feature_tables["item_features"], max_rows=15)
user_item_features_preview_table = df_to_wrapped_table(feature_tables["user_item_features"], max_rows=15)
cooccurrence_preview_table = df_to_wrapped_table(feature_tables["item_cooccurrence"], max_rows=15)

# ============================================================
# 8) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("06 Feature Engineering and Transformation", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(team_table)
story.append(Spacer(1, 14))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents the feature engineering and transformation stage for the recommendation pipeline. It captures feature creation logic, transformation assets, SQL warehouse schema, and transformed outputs stored in a structured database layer.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
story.append(Paragraph("• Create recommendation-ready features such as user activity frequency.", bullet_style))
story.append(Paragraph("• Compute average rating style features per user and item.", bullet_style))
story.append(Paragraph("• Build co-occurrence or similarity-style item pair features.", bullet_style))
story.append(Paragraph("• Store transformed outputs in a structured warehouse database.", bullet_style))
story.append(Paragraph("• Provide SQL schema, transformation script evidence, and summary of feature logic.", bullet_style))
story.append(Spacer(1, 10))

story.append(Paragraph("3. Supporting Project and Log Artifacts", heading_style))
story.append(Paragraph(
    "The following discovered files support this section, including source datasets, validation outputs, and generated warehouse assets. Long paths are wrapped inside table cells using real line breaks at safe separators only.",
    body_style
))
story.append(artifact_table)
story.append(Spacer(1, 12))

story.append(Paragraph("4. Transformation Scripts and SQL Assets", heading_style))
story.append(asset_table)
story.append(Spacer(1, 12))

story.append(Paragraph("5. Feature Logic Summary", heading_style))
all_logic = []
all_logic.extend(event_notes if event_notes else [])
all_logic.extend(product_notes if product_notes else [])
all_logic.extend(feature_tables["feature_logic"] if feature_tables["feature_logic"] else [])

if not all_logic:
    all_logic = ["No feature logic could be inferred because usable project datasets were not found."]

for note in all_logic:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("6. Structured Warehouse Outputs", heading_style))
story.append(transformed_table)
story.append(Spacer(1, 12))

story.append(Paragraph("7. Warehouse and Transformed Folder Structure", heading_style))
story.append(Paragraph("<b>warehouse/</b>", meta_style))
story.append(Preformatted(wrap_block_text(warehouse_tree, width=92), code_style))
story.append(Spacer(1, 8))
story.append(Paragraph("<b>data/transformed/</b>", meta_style))
story.append(Preformatted(wrap_block_text(transformed_tree, width=92), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("8. Feature Table Summary", heading_style))
story.append(feature_summary_table)
story.append(Spacer(1, 12))

story.append(Paragraph("9. User Features Preview", heading_style))
story.append(user_features_preview_table)
story.append(Spacer(1, 12))

story.append(Paragraph("10. Item Features Preview", heading_style))
story.append(item_features_preview_table)
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("11. User-Item Features Preview", heading_style))
story.append(user_item_features_preview_table)
story.append(Spacer(1, 12))

story.append(Paragraph("12. Item Co-occurrence Preview", heading_style))
story.append(cooccurrence_preview_table)
story.append(Spacer(1, 12))

story.append(Paragraph("13. SQL Schema", heading_style))
story.append(Paragraph(
    "The SQL schema below was exported from the structured SQLite warehouse after the transformed tables were created.",
    body_style
))
story.append(Preformatted(wrap_block_text(schema_preview, width=95), code_style))
story.append(Spacer(1, 12))

story.append(PageBreak())

story.append(Paragraph("14. Transformation Code / Asset Preview", heading_style))
if asset_previews:
    for item in asset_previews:
        story.append(Paragraph(f"Asset: {escape(item['name'])}", sub_heading_style))
        story.append(Paragraph(f"<b>Path:</b> {escape(item['relative_path'])}", meta_style))
        story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph("No transformation notebook, script, or SQL preview is available.", body_style))
story.append(Spacer(1, 8))

story.append(Paragraph("15. Validation / Pipeline Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("16. Data Quality Report Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_json_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("17. Validation Event Log Preview", heading_style))
story.append(Preformatted(wrap_block_text(validation_log_preview, width=95), code_style))
story.append(Spacer(1, 10))

story.append(Paragraph("18. Conclusion", heading_style))
story.append(Paragraph(
    "This PDF consolidates the project evidence for feature engineering and transformation, including recommendation features, SQL schema output, transformation asset discovery, and structured warehouse storage suitable for downstream training or feature serving.",
    body_style
))

# ============================================================
# 9) BUILD PDF
# ============================================================
def build_pdf(path):
    doc = SimpleDocTemplate(
        str(path),
        pagesize=A4,
        rightMargin=0.50 * inch,
        leftMargin=0.50 * inch,
        topMargin=0.55 * inch,
        bottomMargin=0.55 * inch,
    )
    doc.build(story)

try:
    build_pdf(OUTPUT_PATH)
    print(f"\nPDF created successfully: {OUTPUT_PATH}")
except PermissionError:
    alt_path = PROJECT_ROOT / f"06 Feature Engineering and Transformation- DM4ML-Group51-{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf"
    build_pdf(alt_path)
    print("\nOriginal PDF is likely open or locked.")
    print(f"Saved alternate file instead: {alt_path}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
RAW_ROOT: C:\Users\barath\recomart-pipeline\data\raw
BRONZE_ROOT: C:\Users\barath\recomart-pipeline\data\bronze
WAREHOUSE DB: C:\Users\barath\recomart-pipeline\warehouse\recommendation_features.db
SCHEMA SQL: C:\Users\barath\recomart-pipeline\warehouse\recommendation_features_schema.sql
OUTPUT PDF: C:\Users\barath\recomart-pipeline\06 Feature Engineering and Transformation- DM4ML-Group51.pdf

PDF created successfully: C:\Users\barath\recomart-pipeline\06 Feature Engineering and Transformation- DM4ML-Group51.pdf
